# Case Management and Caseload Notebook (CRISP-DM)

**Production exports:** Run [`scripts/export_case_onnx.py`](scripts/export_case_onnx.py) to train ONNX models into `../backend/Models/`. Shared helpers live in [`helpers/pipeline_utils.py`](helpers/pipeline_utils.py). See [`case_management_ml/README.md`](case_management_ml/README.md).

This notebook implements three pipelines:
- Predictive Pipeline 1: Risk Management Early-Warning System
- Explanatory Pipeline 2: Drivers of Reintegration Readiness
- Predictive Pipeline 3: NLP Distress Signal Detection


## 1) Business Understanding

Primary operational objective: reduce cases where residents fall through the cracks by flagging short-term regression risk and clarifying reintegration drivers.

## 2) Data Understanding

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

from helpers.pipeline_utils import (
    build_caseload_queue,
    build_nlp_dataset,
    build_reintegration_dataset,
    build_risk_dataset,
    load_tables,
    predict_risk_probabilities,
    resolve_data_root,
    save_artifact,
    split_with_stratify,
    train_nlp_model,
    train_reintegration_model_with_vif,
    train_risk_model,
)

BASE_DIR = Path.cwd()
DATA_ROOT = resolve_data_root(BASE_DIR)
ARTIFACT_DIR = BASE_DIR / "artifacts"

tables = load_tables(DATA_ROOT)
for name, frame in tables.items():
    print(f"{name}: {frame.shape}")


residents: (60, 49)
incidents: (100, 12)
health: (534, 14)
education: (534, 10)
interventions: (180, 11)
process: (2819, 15)
visits: (1337, 14)


## 3) Data Preparation

Feature engineering below uses resident-level aggregations and drops outcome-adjacent leakage fields before training.

In [9]:
x_risk, y_risk = build_risk_dataset(tables)
x_reint, y_reint = build_reintegration_dataset(tables)
x_nlp, y_nlp = build_nlp_dataset(tables)

print("Risk:", x_risk.shape, y_risk.value_counts().to_dict())
print("Reintegration:", x_reint.shape, y_reint.value_counts().to_dict())
print("NLP:", x_nlp.shape, y_nlp.value_counts().to_dict())


Risk: (60, 50) {0: 31, 1: 29}
Reintegration: (60, 50) {0: 41, 1: 19}
NLP: (2819, 3) {0: 2142, 1: 677}


## 4) Modeling

### Pipeline 1: Risk Management Early-Warning (Predictive)
- Model family: ensemble classifier
- Objective metric: Recall (minimize false negatives)
- Split strategy: stratified train/test

In [10]:
risk_split = split_with_stratify(x_risk, y_risk)
risk_model, risk_metrics = train_risk_model(risk_split)

print("Risk Recall:", round(risk_metrics["recall"], 4))
print("Risk Confusion Matrix:\n", risk_metrics["confusion_matrix"])
print(risk_metrics["classification_report"])


Risk Recall: 0.6667
Risk Confusion Matrix:
 [[6 0]
 [2 4]]
              precision    recall  f1-score   support

           0       0.75      1.00      0.86         6
           1       1.00      0.67      0.80         6

    accuracy                           0.83        12
   macro avg       0.88      0.83      0.83        12
weighted avg       0.88      0.83      0.83        12



### Pipeline 2: Drivers of Reintegration Readiness (Explanatory)
- Model family: Logistic Regression
- Statistical rigor: iterative VIF pruning to `VIF <= 5` for numeric explanatory features

In [11]:
reint_split = split_with_stratify(x_reint, y_reint)
reint_model, reint_metrics, selected_cols, final_vif = train_reintegration_model_with_vif(reint_split, threshold=5.0)

print("Reintegration decision threshold:", round(reint_metrics["threshold"], 3))
print("Reintegration Recall:", round(reint_metrics["recall"], 4))
print("Reintegration Confusion Matrix:\n", reint_metrics["confusion_matrix"])
print(reint_metrics["classification_report"])
print("\nRetained explanatory features:")
print(selected_cols)
print("\nFinal VIF values:")
print(final_vif)
print("\nTop positive reintegration drivers:")
print(reint_metrics["top_positive_drivers"])
print("\nTop negative reintegration drivers:")
print(reint_metrics["top_negative_drivers"])


Reintegration decision threshold: 0.2
Reintegration Recall: 0.5
Reintegration Confusion Matrix:
 [[3 5]
 [2 2]]
              precision    recall  f1-score   support

           0       0.60      0.38      0.46         8
           1       0.29      0.50      0.36         4

    accuracy                           0.42        12
   macro avg       0.44      0.44      0.41        12
weighted avg       0.50      0.42      0.43        12


Retained explanatory features:
['safehouse_id', 'safety_concern_rate', 'service_caring_count', 'service_healing_count', 'service_teaching_count', 'service_legal_services_count', 'case_control_no', 'internal_code', 'case_status', 'sex', 'date_of_birth', 'birth_status', 'place_of_birth', 'religion', 'case_category', 'sub_cat_orphaned', 'sub_cat_trafficked', 'sub_cat_child_labor', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse', 'sub_cat_osaec', 'sub_cat_cicl', 'sub_cat_at_risk', 'sub_cat_street_child', 'sub_cat_child_with_hiv', 'is_pwd', 'pwd_type', 'has_

### Pipeline 3: NLP on Process Recordings (Predictive Text Analytics)
- Text feature extraction: TF-IDF
- Structured context: session type + session duration
- Objective: identify concerns and quiet deterioration patterns

In [12]:
nlp_split = split_with_stratify(x_nlp, y_nlp)
nlp_model, nlp_metrics = train_nlp_model(nlp_split)

print("NLP decision threshold:", round(nlp_metrics["threshold"], 3))
print("NLP Recall:", round(nlp_metrics["recall"], 4))
print("NLP Confusion Matrix:\n", nlp_metrics["confusion_matrix"])
print(nlp_metrics["classification_report"])


NLP decision threshold: 0.25
NLP Recall: 0.9926
NLP Confusion Matrix:
 [[  1 428]
 [  1 134]]
              precision    recall  f1-score   support

           0       0.50      0.00      0.00       429
           1       0.24      0.99      0.38       135

    accuracy                           0.24       564
   macro avg       0.37      0.50      0.19       564
weighted avg       0.44      0.24      0.10       564



## 5) Evaluation and Business Translation

Risk score outputs are translated into an actionable caseload queue used to prioritize case conferences.

In [13]:
risk_prob = predict_risk_probabilities(risk_model, risk_split.x_test)
test_resident_ids = tables["residents"].loc[risk_split.x_test.index, "resident_id"]
caseload_queue = build_caseload_queue(test_resident_ids, risk_prob, high_threshold=0.70, medium_threshold=0.45)
caseload_queue.head(15)


,resident_id,risk_probability,priority_band
0,32,0.646676,Elevated - Weekly supervisor review
1,34,0.567934,Elevated - Weekly supervisor review
2,20,0.567775,Elevated - Weekly supervisor review
3,37,0.519518,Elevated - Weekly supervisor review
4,39,0.469338,Elevated - Weekly supervisor review
5,13,0.465333,Elevated - Weekly supervisor review
6,55,0.448114,Routine monitoring
7,28,0.447243,Routine monitoring
8,21,0.393686,Routine monitoring
9,54,0.391643,Routine monitoring


## 6) Deployment Readiness

Serialize pipeline artifacts for .NET/React integration. Input contract:
- `risk_pipeline.joblib` expects tabular resident-level features matching the risk training schema.
- `reintegration_pipeline.joblib` expects selected explanatory features shown above.
- `nlp_pipeline.joblib` expects `session_narrative`, `session_type`, `session_duration_minutes`.

In [14]:
risk_artifact = save_artifact(risk_model, ARTIFACT_DIR, "risk_pipeline.joblib")
reint_artifact = save_artifact(reint_model, ARTIFACT_DIR, "reintegration_pipeline.joblib")
nlp_artifact = save_artifact(nlp_model, ARTIFACT_DIR, "nlp_pipeline.joblib")
queue_output_path = ARTIFACT_DIR / "caseload_priority_queue_sample.csv"
caseload_queue.to_csv(queue_output_path, index=False)

print("Saved:")
print(risk_artifact)
print(reint_artifact)
print(nlp_artifact)
print(queue_output_path)


Saved:
c:\Users\jketh\source\repos\HouseOfHope\ml-pipelines\case_load_test\artifacts\risk_pipeline.joblib
c:\Users\jketh\source\repos\HouseOfHope\ml-pipelines\case_load_test\artifacts\reintegration_pipeline.joblib
c:\Users\jketh\source\repos\HouseOfHope\ml-pipelines\case_load_test\artifacts\nlp_pipeline.joblib
c:\Users\jketh\source\repos\HouseOfHope\ml-pipelines\case_load_test\artifacts\caseload_priority_queue_sample.csv


## 7) QA and Reproducibility Checks

- Deterministic random seeds are set in helper utilities.
- Stratified splits are used for all classification tasks.
- Leakage-adjacent fields are removed in feature builders.
- Notebook is intended to run top-to-bottom and produce serialized artifacts.